In [0]:
spark.sql("DROP CATALOG test CASCADE")

DataFrame[]

In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS edge_pbb.mortage")

DataFrame[]

In [0]:
df = spark.sql("SHOW EXTERNAL LOCATIONS")
display(df)

name,url,comment
adls,abfss://target@adlscarl02.dfs.core.windows.net/databricks_external,
dbrtry,abfss://unity-catalog-storage@dbstorageqjtitx3mdfq44.dfs.core.windows.net/7405615225417288,null


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

# -------------------------
# 1️⃣ 生成日期维度 (覆盖2025-2026)
# -------------------------

date_df = (
    spark.range(0, 730)  # 2年大约730天
    .withColumn("origination_date", 
                F.expr("date_add('2025-01-01', CAST(id AS INT))"))
    .drop("id")
)

# -------------------------
# 2️⃣ 每天生成多条 mortgage 记录
# -------------------------

mortgage_df = (
    date_df
    .withColumn("records_per_day", F.lit(50))  # 每天50笔贷款
    .withColumn("dummy_id", F.explode(F.sequence(F.lit(1), F.col("records_per_day"))))
    .drop("records_per_day")
)

# -------------------------
# 3️⃣ 添加业务字段
# -------------------------

mortgage_df = (
    mortgage_df
    .withColumn("loan_id", F.monotonically_increasing_id())
    .withColumn("customer_id", F.concat(F.lit("CUST_"), F.col("loan_id")))
    .withColumn("principal_amount", 
                F.round(F.rand()*900000 + 100000, 2))  # 100k - 1M
    .withColumn("interest_rate", 
                F.round(F.rand()*3 + 2, 2))  # 2% - 5%
    .withColumn("term_years", 
                F.when(F.rand() < 0.5, 25).otherwise(30))
    .withColumn("property_value",
                F.round(F.col("principal_amount") * (F.rand()*0.3 + 1.1), 2))
    .withColumn("loan_status",
                F.when(F.rand() < 0.85, "ACTIVE")
                 .when(F.rand() < 0.95, "CLOSED")
                 .otherwise("DEFAULT"))
    .withColumn("last_updated_ts", F.current_timestamp())
)

# -------------------------
# 4️⃣ 确认 2026 年覆盖情况
# -------------------------

mortgage_df.filter(F.year("origination_date") == 2026) \
           .select(F.min("origination_date"), F.max("origination_date")) \
           .show()

# -------------------------
# 5️⃣ 写入 Delta 表
# -------------------------

+---------------------+---------------------+
|min(origination_date)|max(origination_date)|
+---------------------+---------------------+
|           2026-01-01|           2026-12-31|
+---------------------+---------------------+



In [0]:
mortgage_df.write \
    .format("delta") \
    .mode("overwrite") \
    .partitionBy("origination_date") \
    .option("path", "abfss://unity-catalog-storage@dbstorageqjtitx3mdfq44.dfs.core.windows.net/7405615225417288/edge_pbb/mortage/dummy_mortgage_historical") \
        
print("Dummy historical mortgage data created successfully!")

Dummy historical mortgage data created successfully!
